# Narratives fMRI dataset

fMRI support is preliminary and limited to one dataset/study, Narratives.
The example below documents this supported path.

Nastase et al. (2021), [OpenNeuro ds002345](https://openneuro.org/datasets/ds002345). Surface-projected BOLD data for
participants listening to stories. Evaluation follows the encoding model
approach: transformer hidden states are used to predict vertex-level BOLD signal
via ridge regression, evaluated by per-vertex Pearson correlation.

## Prerequisites

```bash
pip install -e ".[readers,models]"
replica fetch narratives  # prints the access steps
```

`readers` installs nibabel for the GIfTI surface files, and `models` installs torch and transformers for the GPT-2 encoder.

In [ ]:
from replica.dataset import describe_dataset, load_dataset
from replica.dataset.catalog.registry import REGISTRY

# Story and load-time path/space options are discoverable without loading.
print(describe_dataset("narratives"))

In [ ]:
# Surface-projected GIfTI derivatives inside the dataset directory.
deriv = REGISTRY["narratives"].default_path() / "afni-nosmooth"

ds = load_dataset(
    "narratives",
    story="21styear",
    derivatives_path=str(deriv),
    max_subjects=2,  # limit for tutorial speed
)

In [ ]:
print(f"Stimuli:      {len(ds.stimuli())}")
bold = ds.recordings(measure="bold", modality="fmri")
print(f"BOLD entries: {len(bold)}")
print(f"Participants: {ds.participants}")
print(f"Vertices:     {bold[0].values.shape[1] if bold else 0}")

stim = ds.stimuli()[0]
print(f"\nStimulus ID: {stim.stimulus_id}")
print(f"Words: {len(stim.layers['words'])}")
print(f"First 20 words: {' '.join(stim.unit_strings()[:20])}")

# Word timing layer
times = stim.layers["word_times"]
print("\nWord timing examples:")
for i in range(min(5, len(times))):
    span, (onset, offset) = times[i]
    word = span.extract(stim.text)
    print(f"  {word:15s}  onset={onset:.3f}s  offset={offset:.3f}s")

# BOLD shape
recording = bold[0]
print(f"\nBOLD shape: {recording.values.shape} (samples x vertices)")
print(f"TR: {recording.sample_interval}s")
print(
    f"Trim: start={recording.metadata['trim_start_samples']} samples, end={recording.metadata['trim_end_samples']} samples"
)

In [ ]:
from replica.analysis import EncodingConfig, evaluate
from replica.analysis.specification.evaluation import EvaluationSpec
from replica.feature import HiddenState, HuggingFaceEmbedding, ModelFeature

layers = (0, 6, 11)
model = HuggingFaceEmbedding("gpt2", layers=list(layers))

spec = EvaluationSpec(
    predictors=[ModelFeature(model, HiddenState(layer)) for layer in layers],
    controls=[],
    targets=[ds.feature("bold", modality="fmri")],
)
config = EncodingConfig()
results = evaluate(ds, spec, config)

In [ ]:
print("Encoding model results (GPT-2 -> BOLD prediction):")
print(
    f"{' Layer':>6s}  {'Participant':>15s}  {'Mean r':>8s}  {'Positive/finite':>18s}  {'Undefined':>9s}"
)
print("-" * 72)
for r in results:
    mean_r = r.metrics.get("mean_r", float("nan"))
    n_pos = r.metrics.get("n_positive_channels", 0)
    n_finite = r.metrics.get("n_finite_channels", 0)
    n_undefined = r.metrics.get("n_undefined_channels", 0)
    print(
        f"{r.layer:6d}  {r.participant_id:>15s}  {mean_r:+8.4f}  {n_pos:>7d}/{n_finite}  {n_undefined:>9d}"
    )

## Notes

- Loading BOLD data is memory-intensive, and memory use grows with the number of subjects
- Reading the GIfTI derivatives requires `nibabel` (the `readers` extra); the
  ridge encoding uses scikit-learn, a base dependency
- Use `max_subjects` to limit memory usage during exploration
- Full evaluation with all subjects and multiple layers takes significant time